# Demo Notebook: *On the representations of entities in Auto-Regressive Large Language Models*

This notebook demonstrates the experiments descibed in our papier *On the representations of entities in Auto-Regressive Large Language Models*



## Imports and Utils

In [18]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
#Ensure tto work on the repository path .. 
%cd ~/code/entityrepresentations/

/home/morand/code/entityrepresentations


/home/morand/reimplems/taskVectorsHendel/env/lib/python3.10/site-packages/IPython/core/magics/osm.py:393: UserWarning:

This is now an optional IPython functionality, using bookmarks requires you to install the `pickleshare` library.

/home/morand/reimplems/taskVectorsHendel/env/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning:

This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.



In [34]:
import torch, os, gc, sys
from pathlib import Path
from tqdm import tqdm
# read weights files 

repo_path = Path(os.getcwd())
# load results
res_path = repo_path / "results"
if not res_path.exists():
    raise ValueError("No results found, please make sure your current directory is the repository root")

#get jobs
print(f"Loading results from {res_path}")
results = loadResults(res_path)

import transformer_lens as tl 
from transformer_lens import HookedTransformer, patching

#our own code
import utils 
from LabelExtractor import eval_model, infer_entities
from processResults import *
import circuitsvis as cv
import plotly.io as pio

results.head()


Loading results from /home/morand/code/entityrepresentations/results


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 33/33 [00:00<00:00, 1516.53it/s]


,path,model_name,dataset_name,layer,with_context,extraction_method,max_ent_length,max_length,epochs,logs_per_epoch,lr,batch_size,run,version,history,Eval,date,inference
0,/home/morand/code/entityrepresentations/result...,phi-2,CoNLL2003,4,False,in_context,20,200,10,4,0.04,30,4.0,1.0,"[{'epoch': 0.2490118577075099, 'loss': 1.36271...","{'Partial Match': 0.6901094822787159, 'Exact M...",1.739180e+09,/home/morand/code/entityrepresentations/result...
1,/home/morand/code/entityrepresentations/result...,phi-2,CoNLL2003,28,False,in_context,20,200,10,4,0.04,30,1.0,1.0,"[{'epoch': 0.2490118577075099, 'loss': 1.35745...","{'Partial Match': 0.6607904991649657, 'Exact M...",1.739180e+09,/home/morand/code/entityrepresentations/result...
2,/home/morand/code/entityrepresentations/result...,phi-2,CoNLL2003,6,False,in_context,20,200,10,4,0.04,30,0.0,1.0,"[{'epoch': 0.2490118577075099, 'loss': 1.41423...","{'Partial Match': 0.6838003340137316, 'Exact M...",1.739180e+09,/home/morand/code/entityrepresentations/result...
3,/home/morand/code/entityrepresentations/result...,phi-2,CoNLL2003,23,False,in_context,20,200,10,4,0.04,30,1.0,1.0,"[{'epoch': 0.2490118577075099, 'loss': 1.17705...","{'Partial Match': 0.6850992763035814, 'Exact M...",1.739180e+09,/home/morand/code/entityrepresentations/result...
4,/home/morand/code/entityrepresentations/result...,phi-2,CoNLL2003,26,False,in_context,20,200,10,4,0.04,30,0.0,1.0,"[{'epoch': 0.2490118577075099, 'loss': 1.22254...","{'Partial Match': 0.677120059380219, 'Exact Ma...",1.739180e+09,/home/morand/code/entityrepresentations/result...


In [ ]:
### Testing the library for plotting

# Plotly needs a different renderer for VSCode/Notebooks vs Colab argh
pio.renderers.default = "notebook_connected"
print(f"Using renderer: {pio.renderers.default}")
# Testing that the library works
cv.examples.hello("Fellow AI researcher")


## Params 
Thanks to the [`transformer_lens`](https://github.com/TransformerLensOrg/TransformerLens/tree/main) library, we can load and use many different llms seemlessly.

In [35]:
# Load a model (eg GPT-2 Small)
model_name = "meta-llama/Meta-Llama-3-8B" # ?
model_name = "gpt2-small" # 117M ok
model_name = "pythia-2.8b"#ok !
model_name = "gpt2-xl" # 1.5B ok
model_name = "mistralai/Mistral-7B-v0.1" # ok JZ a100 8cpus | 
model_name = "gpt2-large" # 774M ok
model_name = "gpt2-medium" # 302M ok
model_name = "phi-1_5"  # 1.5B ok
model_name = "phi-2" # 2,5B ok 12cpus nope, gpu 24cpus ok

with_context = True
with_context = False


## Load Model

In [38]:
#check if model variable exists
if not 'model' in locals():
    model = utils.load_model(model_name)
    dim = model.QK.shape[-1]
print(model)
model.eval()
model = model.cuda()

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loaded pretrained model phi-2 into HookedTransformer
HookedTransformer(
  (embed): Embed()
  (hook_embed): HookPoint()
  (blocks): ModuleList(
    (0-31): 32 x TransformerBlock(
      (ln1): LayerNorm(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (ln2): LayerNorm(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (attn): Attention(
        (hook_k): HookPoint()
        (hook_q): HookPoint()
        (hook_v): HookPoint()
        (hook_z): HookPoint()
        (hook_attn_scores): HookPoint()
        (hook_pattern): HookPoint()
        (hook_result): HookPoint()
        (hook_rot_k): HookPoint()
        (hook_rot_q): HookPoint()
      )
      (mlp): MLP(
        (hook_pre): HookPoint()
        (hook_post): HookPoint()
      )
      (hook_attn_in): HookPoint()
      (hook_q_input): HookPoint()
      (hook_k_input): HookPoint()
      (hook_v_input): HookPoint()
      (hook_mlp_in): HookPoint()
      (hook_attn_out

## Load the data

In [39]:
dataset_name = "CoNLL2003"
train_dataset, test_dataset, val_dataset = utils.load_datasets(dataset_name, max_ent_length=200)

In [40]:
print("train length:", len(train_dataset))
print("dev length:", len(val_dataset))
print("test length:", len(test_dataset))
print("ex sample:") 
item = train_dataset[np.random.randint(len(val_dataset))]
for key in item.keys():
    print(" -", key, ":", item[key])

train length: 22749
dev length: 5695
test length: 5389
ex sample:
 - entity : NZ
 - text : BNZ cuts NZ fixed home lending rates.
 - id : 3010


## Test TaskVecs

In [41]:
layer = 10 # Layer at which we want to extract the trained task vector

fileName = get_taskVec(results, model_name, layer=layer, dataset_name=dataset_name, with_context = with_context)

TaskVec = torch.load(fileName, weights_only=True)
print("TaskVec loaded from ", fileName)

found 1 jobs for layer 10 of phi-2 without context on CoNLL2003 with method in_context.
found ['TaskVec_phi-2_l10_e10.0.pth'] 
TaskVec loaded from  /home/morand/code/entityrepresentations/results/67284b69fb65b648fa447911c12daa0885c1b7a0f79d1088b0544bf401f65e43/TaskVec_phi-2_l10_e10.0.pth


# Entity Lens
Now that we have a LLM and a trained task vector $\theta_\ell$ loaded, we can infer entities from any representations.
We showcase here the *Entity Lens*, that generates a mention for each token considered at specifed layers, allowing to visualize to what *entity* the model is *thinking* in its internal representations.

In [42]:
ind = np.random.randint(len(test_dataset))
# ind = 7616
context = test_dataset[ind]["text"]
# context = " _ > Agnes Kant"
# context = "Alan Bean was an American astronaut, born on March 15, 1932 in Wheeler, Texas. He received a Bachelor of Science degree at the University of Texas at Austin in 1955 and was chosen by NASA in 1963. _ > Alan Bean"
# context = ...

words = model.to_str_tokens(context)
print(len(words))
cv.tokens.colored_tokens(words, words)


35


In [43]:
#compute whole cache
print("computing cache ...")
#get whole hidden states
_ , cache = model.run_with_cache(context)
repr = cache[tl.utils.get_act_name("resid_post", layer, "")][0,:,:] # 1 x n_tokens x dim
repr = repr.detach().cuda()
print(repr.shape)
data = [
    {   
        "id": i,
        "text": context,
        "representation": repr[i],
        "tok": words[i],
    }
    for i in range(len(words))
]
infer_entities(model, TaskVec, data, with_context=with_context)

computing cache ...
torch.Size([35, 2560])


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:04<00:00,  1.05s/it]


### Raw display

In [44]:
print(data[0]["text"])
print("Token".center(20), "| Inferred")
print("-"*60)
for it in data:
    print(f"{it['tok'].center(20)} | {it['inferred']}" )

International trade union leaders on Friday expressed outrage that the head of the International Labour Organisation (ILO) had been barred from speaking at next week's WTO meeting in Singapore.
       Token         | Inferred
------------------------------------------------------------
   <|endoftext|>     | Los Angeles
   International     | International
        trade        | International trade
        union        | International trade union
       leaders       | trade union leaders
         on          | U.S. senators on
       Friday        | Friday
      expressed      | opposition expressed
       outrage       | outrage
        that         | U.S. Olympic Committee that the U.S. Olympic Committee that the U.S.
         the         | U.S. West Coast
        head         | head
         of          | head
         the         | president
    International    | International
       Labour        | International Labour
    Organisation     | World Health Organisation
          (

### Circuitvis
smoother visualization

In [45]:
html = cv.tokens.colored_tokens(words, [it['inferred'] for it in data])
html.cdn_src = html.cdn_src.replace("margin: 15px", "margin: 50px")
html

### Old IPywidget code

In [46]:
from IPython.display import display
import ipywidgets as widgets

prompt = train_dataset[np.random.randint(len(train_dataset))]["text"]
# prompt = " _ > Agnes Kant"
# prompt = "Alan Bean was an American astronaut, born on March 15, 1932 in Wheeler, Texas. He received a Bachelor of Science degree at the University of Texas at Austin in 1955 and was chosen by NASA in 1963. _ > Alan Bean"
words = model.to_str_tokens(prompt)
print(words)
print(len(words))

# Callback function to update selected word
def on_button_click(b):
    selected_word_label.value = f"Selected Word: {b.description}"
    layer = dropdown.value
    repr = utils.get_representation(model,
                                layer=layer,
                                tokens=model.to_tokens(prompt),
                                token_inds=torch.tensor([b.id]),
                                verbose=True)
    #change button color
    b.style.button_color = 'lightgreen'
    #change all others to default
    for button in buttons:
        if button.id != b.id:
            button.style.button_color = 'white'
    data = [{
        "id":0,
        "representation": repr,
        "text": prompt
        }]
    infer_entities(model, TaskVec, data, with_context=with_context)
    print("generation:", data[0]["inferred"])

# Buttons for each word
buttons = []
class clickableToken(widgets.Button):
    def __init__(self, id:int, description:str, **kwargs):
        super().__init__(description=description, **kwargs)
        self.description = description
        self.id = id
        self.on_click(on_button_click)

for ind, token in enumerate(words):
    buttons.append(
        clickableToken(
            id=ind, 
            description=token,
            layout=widgets.Layout(width='auto', margin='2px', padding='0 5px')
        ))
    

# Box widget with flexible wrapping
button_box = widgets.Box(
    children=buttons,
    layout=widgets.Layout(display='flex', flex_flow='row wrap', align_items='center')
)
# Dropdown widget for selecting an integer
dropdown = widgets.Dropdown(
    options=[(f"layer {i}", i) for i in range(len(model.blocks)-1, -1, -1)],
    description='Select a layer:',
    disabled=False,    
)
dropdown.value = layer
def on_checkbox_change(change):
    global with_context
    with_context = change['new']

# Create the checkbox widget
with_context_checkbox = widgets.Checkbox(
    value=with_context,
    description='Generate with context',
    disabled=False
)
# Link the function to the checkbox change event
with_context_checkbox.observe(on_checkbox_change, names='value')

# Label to display the selected word
selected_word_label = widgets.Label()

# # clean previous display
# display.clear_output() #works ?? 

# Create a title using HTML widget
title = widgets.HTML(value="<h3>Select a Token to generate from:</h3>")

# Display widgets
display(title)
display(button_box)
#display checkbox  and dropdown side by side
display(widgets.HBox([with_context_checkbox, dropdown]))
display(selected_word_label)

['<|endoftext|>', 'Inter', 'fax', ',', ' quoting', ' Yel', 'ts', 'in', ' press', ' secretary', ' Sergei', ' Y', 'ast', 'r', 'z', 'hem', 'bs', 'ky', ',', ' said', ' Yel', 'ts', 'in', ' and', ' Koh', 'l', ' had', ' discussed', ' bilateral', ' relations', ' and', ' international', ' issues', ' on', ' the', ' telephone', '.']
37


HTML(value='<h3>Select a Token to generate from:</h3>')

Box(children=(clickableToken(description='<|endoftext|>', layout=Layout(margin='2px', padding='0 5px', width='…

Label(value='')